# Genesis — IE 643 | Day 1: Setup, Benchmark, Baseline & Method A
Model: **Wan2.2 TI2V-5B** (flow matching, unified T2V + I2V)

**Before running:** Settings panel → Accelerator: **GPU T4 x2** · Internet: **On**

Run the cells **in order, one at a time**. Stop after the benchmark cell (Step 7) and check the timing before running Step 8.

Everything that should survive the session is written to `/kaggle/working/genesis/`.

## Step 1 — Check hardware and disk

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /tmp /kaggle/working | cat
!free -g | head -2

## Step 2 — Install libraries (~2 min)

In [ ]:
!pip uninstall -y -q torchao   # Kaggle's preinstalled torchao is too old for current diffusers; we do not use it
!pip install -q -U diffusers transformers accelerate ftfy imageio imageio-ffmpeg huggingface_hub
import diffusers, transformers, torch
print("diffusers", diffusers.__version__, "| transformers", transformers.__version__, "| torch", torch.__version__)
print("GPUs:", torch.cuda.device_count())

## Step 3 — Configuration
All settings live here. Change resolution / frames / steps here only.

In [ ]:
import os, json, time, gc, shutil, re, html   # (rerun this cell after any kernel restart)
import numpy as np
import torch

os.environ["HF_HOME"] = "/tmp/hf_home"           # model cache on temporary disk, NOT /kaggle/working
MODEL_ID    = "Wan-AI/Wan2.2-TI2V-5B-Diffusers"
WEIGHTS_DIR = "/tmp/wan22_ti2v5b"
OUT         = "/kaggle/working/genesis"
for d in ["embeds", "videos", "latents", "frames", "logs"]:
    os.makedirs(f"{OUT}/{d}", exist_ok=True)

HEIGHT, WIDTH = 480, 832    # must be multiples of 32 for TI2V-5B
NUM_FRAMES    = 49          # must be 4k+1; 49 frames ~ 2 s at 24 fps
FPS           = 24
STEPS         = 30          # denoising steps (model default is 50)
GUIDANCE      = 5.0         # CFG scale
FLOW_SHIFT    = 3.0         # lower shift suits lower resolution
SEED          = 42

NEG_PROMPT = ("Bright tones, overexposed, static, blurred details, subtitles, style, works, paintings, images, static, "
              "overall gray, worst quality, low quality, JPEG compression residue, ugly, incomplete, extra fingers, "
              "poorly drawn hands, poorly drawn faces, deformed, disfigured, misshapen limbs, fused fingers, still picture, "
              "messy background, three legs, many people in the background, walking backwards")

PROMPT_SEQS = {
    "fox": [
        "A red fox with a bushy white-tipped tail walks slowly through a snowy pine forest at sunrise, soft golden light, cinematic, steady tracking shot.",
        "The same red fox stops beside a frozen stream and sniffs the snow, soft golden sunrise light in the snowy pine forest, steady tracking shot.",
        "The same red fox jumps across the frozen stream and trots away between the pine trees, soft golden sunrise light, steady tracking shot.",
    ],
    "boat": [
        "A small white paper boat floats along a rain-filled street gutter at dusk, neon shop signs reflecting in the water, close-up, cinematic.",
        "The same white paper boat spins gently as it drifts past a storm drain, neon reflections rippling on the water, close-up, cinematic.",
        "The same white paper boat drifts into a wide puddle under a streetlight as rain keeps falling, neon reflections, close-up, cinematic.",
    ],
}
json.dump({"HEIGHT": HEIGHT, "WIDTH": WIDTH, "NUM_FRAMES": NUM_FRAMES, "FPS": FPS, "STEPS": STEPS,
           "GUIDANCE": GUIDANCE, "FLOW_SHIFT": FLOW_SHIFT, "SEED": SEED, "PROMPT_SEQS": PROMPT_SEQS},
          open(f"{OUT}/logs/config_day1.json", "w"), indent=2)
print("Config saved.")

## Step 4 — Encode all prompts once with the text encoder (UMT5-XXL), then delete it
The text encoder is too large to keep on a T4 next to the 5B model. We run it once, save the embeddings,
and free both GPU memory and disk. With 2 GPUs it runs in fp32 split across both; otherwise on CPU (slower, a few minutes).

In [ ]:
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer, UMT5EncoderModel
import ftfy

snapshot_download(MODEL_ID, local_dir=WEIGHTS_DIR,
                  allow_patterns=["model_index.json", "tokenizer/*", "text_encoder/*"])

tok = AutoTokenizer.from_pretrained(f"{WEIGHTS_DIR}/tokenizer")
if torch.cuda.device_count() >= 2:
    te = UMT5EncoderModel.from_pretrained(f"{WEIGHTS_DIR}/text_encoder", torch_dtype=torch.float32,
                                          device_map="auto", max_memory={0: "14GiB", 1: "14GiB", "cpu": "20GiB"})
    te_device = "cuda:0"
else:
    te = UMT5EncoderModel.from_pretrained(f"{WEIGHTS_DIR}/text_encoder", torch_dtype=torch.bfloat16)
    te_device = "cpu"
te.eval()

def prompt_clean(t):
    t = ftfy.fix_text(t)
    t = html.unescape(html.unescape(t)).strip()
    return re.sub(r"\s+", " ", t).strip()

@torch.no_grad()
def t5_embed(text, max_len=512):
    # Mirrors diffusers' Wan text encoding: keep real tokens, zero the padding positions.
    enc = tok([prompt_clean(text)], padding="max_length", max_length=max_len, truncation=True,
              add_special_tokens=True, return_attention_mask=True, return_tensors="pt")
    ids, mask = enc.input_ids.to(te_device), enc.attention_mask.to(te_device)
    n = int(mask.sum())
    h = te(ids, mask).last_hidden_state[0, :n].float().cpu()
    out = torch.zeros(max_len, h.shape[-1])
    out[:n] = h
    return out.unsqueeze(0)            # [1, 512, 4096]

t0 = time.time()
embeds = {"__negative__": t5_embed(NEG_PROMPT)}
for name, prompts in PROMPT_SEQS.items():
    for i, p in enumerate(prompts):
        embeds[f"{name}_{i}"] = t5_embed(p)
torch.save(embeds, f"{OUT}/embeds/day1_embeds.pt")
print(f"Encoded {len(embeds)} prompts in {time.time()-t0:.0f}s ->", f"{OUT}/embeds/day1_embeds.pt")

# Free the text encoder from GPU/RAM and disk
del te; gc.collect(); torch.cuda.empty_cache()
shutil.rmtree(f"{WEIGHTS_DIR}/text_encoder", ignore_errors=True)
shutil.rmtree("/tmp/hf_home", ignore_errors=True)
!df -h /tmp | cat

## Step 5 — Download the transformer, VAE and scheduler

In [ ]:
snapshot_download(MODEL_ID, local_dir=WEIGHTS_DIR,
                  allow_patterns=["model_index.json", "transformer/*", "vae/*", "scheduler/*"])
!du -sh {WEIGHTS_DIR}/* | cat

## Step 6 — Build the T2V and I2V pipelines (sharing one transformer and one VAE)
- Transformer in **fp16** (T4 has no native bf16), VAE in **fp32** (better decoding).
- `expand_timesteps` comes from the model's own config: this is the TI2V mechanism where the conditioning
  frame's tokens get timestep 0 (clean) while the rest are noisy.
- Model CPU offload keeps only the module in use on the GPU.

In [ ]:
import diffusers
from diffusers import WanPipeline, WanImageToVideoPipeline, AutoencoderKLWan, WanTransformer3DModel
from diffusers.utils import export_to_video
from PIL import Image

model_index = json.load(open(f"{WEIGHTS_DIR}/model_index.json"))
EXPAND = bool(model_index.get("expand_timesteps", False))
print("expand_timesteps from model config:", EXPAND)

vae = AutoencoderKLWan.from_pretrained(f"{WEIGHTS_DIR}/vae", torch_dtype=torch.float32)
transformer = WanTransformer3DModel.from_pretrained(f"{WEIGHTS_DIR}/transformer", torch_dtype=torch.float16)

sched_cfg = json.load(open(f"{WEIGHTS_DIR}/scheduler/scheduler_config.json"))
SchedCls = getattr(diffusers, sched_cfg["_class_name"])
scheduler = SchedCls.from_pretrained(f"{WEIGHTS_DIR}/scheduler")
if "flow_shift" in scheduler.config:
    scheduler = SchedCls.from_config(scheduler.config, flow_shift=FLOW_SHIFT)
elif "shift" in scheduler.config:
    scheduler = SchedCls.from_config(scheduler.config, shift=FLOW_SHIFT)
print("Scheduler:", type(scheduler).__name__)

t2v = WanPipeline(tokenizer=None, text_encoder=None, vae=vae, scheduler=scheduler,
                  transformer=transformer, expand_timesteps=EXPAND)
i2v = WanImageToVideoPipeline(tokenizer=None, text_encoder=None, vae=vae, scheduler=scheduler,
                              transformer=transformer, expand_timesteps=EXPAND)
# I2V encodes the conditioning image with the VAE *before* denoising; this order lets the
# transformer offload the VAE when it starts, so both never sit on the 16 GB GPU together.
i2v.model_cpu_offload_seq = "vae->transformer"

def activate(p):
    for q in (t2v, i2v):
        try: q.remove_all_hooks()
        except Exception: pass
    p.enable_model_cpu_offload(gpu_id=0)

@torch.no_grad()
def decode(p, latents):
    v = p.vae
    lat = latents.to(v.dtype)
    mean = torch.tensor(v.config.latents_mean).view(1, v.config.z_dim, 1, 1, 1).to(lat.device, lat.dtype)
    inv_std = 1.0 / torch.tensor(v.config.latents_std).view(1, v.config.z_dim, 1, 1, 1).to(lat.device, lat.dtype)
    lat = lat / inv_std + mean
    video = v.decode(lat, return_dict=False)[0]
    frames = p.video_processor.postprocess_video(video, output_type="np")[0]   # (F, H, W, 3) in [0, 1]
    p.maybe_free_model_hooks()
    return frames

E = torch.load(f"{OUT}/embeds/day1_embeds.pt")
def emb(key):  return E[key].to("cuda", torch.float16)

def _common(seed):
    return dict(negative_prompt_embeds=emb("__negative__"), height=HEIGHT, width=WIDTH,
                num_frames=NUM_FRAMES, num_inference_steps=STEPS, guidance_scale=GUIDANCE,
                generator=torch.Generator("cpu").manual_seed(seed), output_type="latent")

def check(frames, tag):
    if (not np.isfinite(frames).all()) or frames.max() < 0.02:
        print(f"WARNING [{tag}]: black or NaN frames — likely fp16 overflow. Tell me before continuing.")

def gen_t2v(key, seed):
    activate(t2v); torch.cuda.reset_peak_memory_stats(); t0 = time.time()
    lat = t2v(prompt_embeds=emb(key), **_common(seed)).frames
    frames = decode(t2v, lat)
    stats = {"sec": round(time.time() - t0, 1), "peak_gb": round(torch.cuda.max_memory_allocated() / 2**30, 2)}
    check(frames, key)
    return frames, lat.cpu().half(), stats

def gen_i2v(image, key, seed):
    activate(i2v); torch.cuda.reset_peak_memory_stats(); t0 = time.time()
    lat = i2v(image=image, prompt_embeds=emb(key), **_common(seed)).frames
    frames = decode(i2v, lat)
    stats = {"sec": round(time.time() - t0, 1), "peak_gb": round(torch.cuda.max_memory_allocated() / 2**30, 2)}
    check(frames, key)
    return frames, lat.cpu().half(), stats

def to_pil(frame):  return Image.fromarray((np.clip(frame, 0, 1) * 255).round().astype("uint8"))

def save_clip(frames, lat, tag):
    export_to_video(list(frames), f"{OUT}/videos/{tag}.mp4", fps=FPS)
    torch.save(lat, f"{OUT}/latents/{tag}.pt")
    # keep the last 17 frames losslessly — Method B (Day 2) re-encodes these as multi-frame context
    np.save(f"{OUT}/frames/{tag}_tail17.npy", (np.clip(frames[-17:], 0, 1) * 255).round().astype("uint8"))

print("Pipelines ready.")

## Step 7 — Benchmark: generate ONE clip and measure it
**Stop here.** Watch `videos/bench_fox_0.mp4` and send me the printed time and peak memory.

In [ ]:
frames, lat, stats = gen_t2v("fox_0", SEED)
save_clip(frames, lat, "bench_fox_0")
print("Benchmark:", stats, "| frames:", frames.shape, "| latents:", tuple(lat.shape))
json.dump(stats, open(f"{OUT}/logs/benchmark.json", "w"))
n_clips_day1 = sum(len(v) * 2 - 1 for v in PROMPT_SEQS.values())
print(f"Step 8 will generate {n_clips_day1} clips ≈ {n_clips_day1 * stats['sec'] / 60:.0f} min")

## Step 8 — Baseline vs Method A on both prompt sequences
- **Baseline:** every clip generated independently from text (T2V), joined with hard cuts.
- **Method A:** clip 1 is shared; each later clip is I2V, conditioned on the previous clip's last frame.
  Its first frame duplicates that conditioning frame, so it is dropped when stitching.
- Same prompt and seed per clip index in both methods, so the only difference is the chaining.

In [ ]:
log = []
for name, prompts in PROMPT_SEQS.items():
    seeds = [SEED + i for i in range(len(prompts))]

    # Baseline
    base = []
    for i in range(len(prompts)):
        f, l, s = gen_t2v(f"{name}_{i}", seeds[i]); save_clip(f, l, f"{name}_baseline_c{i}")
        base.append(f); log.append({"seq": name, "method": "baseline", "clip": i, **s}); print(log[-1])
    export_to_video(list(np.concatenate(base, axis=0)), f"{OUT}/videos/{name}_BASELINE_full.mp4", fps=FPS)

    # Method A (clip 0 reused from the baseline)
    chain = [base[0]]
    for i in range(1, len(prompts)):
        f, l, s = gen_i2v(to_pil(chain[-1][-1]), f"{name}_{i}", seeds[i]); save_clip(f, l, f"{name}_methodA_c{i}")
        chain.append(f[1:]); log.append({"seq": name, "method": "A_lastframe", "clip": i, **s}); print(log[-1])
    export_to_video(list(np.concatenate(chain, axis=0)), f"{OUT}/videos/{name}_METHOD_A_full.mp4", fps=FPS)

    json.dump(log, open(f"{OUT}/logs/day1_runs.json", "w"), indent=2)

print("Done. Full videos:", sorted(x for x in os.listdir(f"{OUT}/videos") if x.endswith("_full.mp4")))

## What to send back
1. The benchmark line from Step 7 (seconds, peak GB).
2. `logs/day1_runs.json`.
3. Your impression of `*_BASELINE_full.mp4` vs `*_METHOD_A_full.mp4` — especially at the seams.
4. Any error message, copied in full.